In [1]:
import polars as pl


pdb_af3 = pl.read_parquet("../../data/pdb/triad/staged/pdb_triad.af3_rmsd.parquet")

In [4]:
pdb_af3.filter(pl.col("pdb") == "8trl")

job_name,cognate,peptide,mhc_class,mhc_1_chain,mhc_1_species,mhc_1_name,mhc_1_seq,mhc_2_chain,mhc_2_species,mhc_2_name,mhc_2_seq,tcr_1_chain,tcr_1_species,tcr_1_seq,tcr_2_chain,tcr_2_species,tcr_2_seq,tcr_1_cdr_1,tcr_1_cdr_2,tcr_1_cdr_2_5,tcr_1_cdr_3,tcr_2_cdr_1,tcr_2_cdr_2,tcr_2_cdr_2_5,tcr_2_cdr_3,pdb,pdb_date,replication,peptide_segid,mhc_1_segid,mhc_2_segid,tcr_1_segid,tcr_2_segid,cdr_rmsd,cdr_rmsd_af2_full,cdr_rmsd_af2_trim,…,pred_mhc_axes_3,pred_mhc_origin_3,pred_tcr_axes_3,pred_tcr_origin_3,pred_dgeom_3,pred_sample_rank_3,pred_mhc_axes_4,pred_mhc_origin_4,pred_tcr_axes_4,pred_tcr_origin_4,pred_dgeom_4,pred_sample_rank_4,docking_rmsd_af3_0,cdr_rmsd_af3_0,peptide_rmsd_af3_0,mhc_rmsd_af3_0,tcr_rmsd_af3_0,docking_rmsd_af3_1,cdr_rmsd_af3_1,peptide_rmsd_af3_1,mhc_rmsd_af3_1,tcr_rmsd_af3_1,docking_rmsd_af3_2,cdr_rmsd_af3_2,peptide_rmsd_af3_2,mhc_rmsd_af3_2,tcr_rmsd_af3_2,docking_rmsd_af3_3,cdr_rmsd_af3_3,peptide_rmsd_af3_3,mhc_rmsd_af3_3,tcr_rmsd_af3_3,docking_rmsd_af3_4,cdr_rmsd_af3_4,peptide_rmsd_af3_4,mhc_rmsd_af3_4,tcr_rmsd_af3_4
str,bool,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,"datetime[μs, UTC]",bool,str,str,str,str,str,f64,f64,f64,…,list[list[f64]],list[f64],list[list[f64]],list[f64],struct[8],i64,list[list[f64]],list[f64],list[list[f64]],list[f64],struct[8],i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64


In [9]:
import requests
from pathlib import Path

url = "https://files.rcsb.org/download/"

for row in pdb_af3.iter_rows(named=True):
    id = row["pdb"]

    r = requests.get(url + id.upper() + ".pdb", allow_redirects=True)

    with open("dump" + "/" + id.upper() + ".pdb", "wb") as f:
        f.write(r.content)

In [27]:
import MDAnalysis as mda
from MDAnalysis.lib.util import convert_aa_code
from collections import defaultdict

invalid = []

for row in pdb_af3.iter_rows(named=True):
    id = row["pdb"]
    # print(id)

    try:
        u = mda.Universe("dump/" + id.upper() + ".pdb")
    except Exception:
        u = mda.Universe("dump/" + id.upper() + ".cif")

    d = {"pdb": id, "invalid": []}

    for res in u.select_atoms("not water").residues:

        try:
            convert_aa_code(res.resname)
        except Exception:
            # print(res.resname)
            d["invalid"].append(res.resname)

    if len(d["invalid"]):
        invalid.append(d)

invalid = pl.DataFrame(invalid)

/tgen_labs/altin/miniconda3/envs/tcrtrifold-experiments/lib/python3.12/site-packages/MDAnalysis/topology/PDBParser.py:384: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
/tgen_labs/altin/miniconda3/envs/tcrtrifold-experiments/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:465: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"


In [41]:
invalid

pdb,invalid
str,list[str]
"""4may""","[""SO4""]"
"""4p4k""","[""NAG"", ""NAG"", … ""0BE""]"
"""6dfs""","[""NAG""]"
"""4grl""","[""NAG""]"
"""6dfx""","[""NAG"", ""NAG"", … ""NAG""]"
…,…
"""3ffc""","[""CD"", ""CL"", … ""CL""]"
"""3gsn""","[""SO4"", ""CL""]"
"""4ozf""","[""NAG"", ""NAG""]"


In [40]:
from xlsxwriter import Workbook

with Workbook("invalid_codes.xlsx") as wb:

    invalid.sort(by="pdb").write_excel(workbook=wb, worksheet="invalid_pdb")
    invalid.explode("invalid").select("invalid").unique().write_excel(
        workbook=wb, worksheet="unique_invalid"
    )